# SVM
### The model is written from scratch in PyTorch in `self_module/svm.py` (the maths is in the comments there).

In [1]:
import sys
import numpy as np
import pandas as pd
import torch
import plotly.express as px
from sklearn.metrics import log_loss , roc_auc_score , average_precision_score , accuracy_score , precision_score , recall_score , f1_score , confusion_matrix , roc_curve

sys.path.append('../..')
from self_module import WeightedSVM

In [2]:
import plotly.io as pio
pio.templates["plotly_dark"].layout.paper_bgcolor = "black"
pio.templates["plotly_dark"].layout.plot_bgcolor = "black"
pio.templates.default = "plotly_dark"

## 3. Machine Learning (SVM)
### Why SVM : it finds the hyperplane that separates clicks and non clicks with the widest margin. Only the points near it matter.
### Same straight line as logistic regression with a different loss , so it is a fair comparison.
### Linear SVM : a kernel SVM needs an n x n matrix , and 1 million x 1 million is far too big.
### Limitations : it gives a score , not a probability , so no log loss and no direct price for an impression. C has to be tuned.
### Class weighted (WeightedSVM) from the start : in the bank assignment the plain hinge called almost everyone "no".

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def load(name):
    X = torch.tensor(pd.read_parquet(f'../../data/ctr/X_{name}.parquet').values , dtype=torch.float32)
    y = torch.tensor(pd.read_parquet(f'../../data/ctr/y_{name}.parquet').values , dtype=torch.float32)
    X = torch.cat([X , torch.ones((len(X) , 1))] , dim=1)
    return X.to(device) , y.to(device)

X_train , y_train = load('train')
X_val , y_val = load('val')
X_test , y_test = load('test')
y_train_np , y_val_np , y_test_np = y_train.cpu().numpy().ravel() , y_val.cpu().numpy().ravel() , y_test.cpu().numpy().ravel()

### The learning rate is lowered step by step during training , otherwise the SVM loss keeps jumping around the minimum.

In [4]:
def fit(model):
    model.epochs = 500
    for lr in [0.1 , 0.03 , 0.01 , 0.003]:
        model.lr = lr
        model.train(X_train , y_train)
    return model

def score(model , X):
    return model.forward(X).cpu().numpy().ravel()

### Tuning C on validation (bigger C = wider margin = simpler model).

In [5]:
models , auc = {} , {}
for C in [0.001 , 0.01 , 0.03 , 0.1]:
    models[C] = fit(WeightedSVM(C=C))
    auc[C] = roc_auc_score(y_val_np , score(models[C] , X_val))
best_C = pd.Series(auc).idxmax()
print('best C :' , best_C)
pd.Series(auc).round(4)

best C : 0.01


0.001    0.7021
0.010    0.7069
0.030    0.7049
0.100    0.7009
dtype: float64

In [6]:
model = models[best_C]
train_loss = [h[0] for h in model.history]       # history holds (train loss , val loss)
fig = px.line(y=train_loss , labels={'x' : 'epoch' , 'y' : 'loss'} , title='Loss')
fig.show()

### The loss drops fast and then flattens (0.711 after 500 epochs , 0.704 at the end).

## Evaluation
### Picking the threshold on the score with the best F1 on validation.

In [7]:
z_val = score(model , X_val)
thresholds = np.arange(-1 , 1.5 , 0.01)
f1s = [f1_score(y_val_np , z_val >= t) for t in thresholds]
threshold = thresholds[np.argmax(f1s)]
threshold

np.float64(0.7300000000000015)

In [8]:
def scores(y , z , pred):
    return {'roc_auc' : roc_auc_score(y , z) , 'pr_auc' : average_precision_score(y , z) ,
            'accuracy' : accuracy_score(y , pred) , 'precision' : precision_score(y , pred , zero_division=0) ,
            'recall' : recall_score(y , pred) , 'f1' : f1_score(y , pred)}

z_train , z_test = score(model , X_train) , score(model , X_test)
pd.DataFrame({
    'val'   : scores(y_val_np , z_val , z_val >= threshold) ,
    'train' : scores(y_train_np , z_train , z_train >= threshold) ,
    'test'  : scores(y_test_np , z_test , z_test >= threshold) ,
}).round(3)

,val,train,test
roc_auc,0.707,0.708,0.704
pr_auc,0.312,0.311,0.312
accuracy,0.622,0.622,0.621
precision,0.268,0.266,0.266
recall,0.698,0.697,0.693
f1,0.387,0.385,0.384


### ROC-AUC 0.704 on test. Train , val and test are within 0.004 , so no overfitting.
### The threshold (0.73) catches 69% of the clicks at 27% precision.

In [9]:
cm = confusion_matrix(y_test_np , z_test >= threshold)
fig = px.imshow(cm , text_auto=True , x=['predicted no click' , 'predicted click'] , y=['actual no click' , 'actual click'] , title='Confusion matrix (test)')
fig.show()

### Simple sklearn implementation

In [10]:
from sklearn.linear_model import SGDClassifier

sk_model = SGDClassifier(loss='hinge' , alpha=best_C , class_weight='balanced' , random_state=42)
sk_model.fit(pd.read_parquet('../../data/ctr/X_train.parquet') , y_train_np)
z_sk = sk_model.decision_function(pd.read_parquet('../../data/ctr/X_test.parquet'))

### Our model vs sklearn

In [11]:
pd.DataFrame({
    'our model' : scores(y_test_np , z_test , z_test >= threshold) ,
    'sklearn'   : scores(y_test_np , z_sk , z_sk >= threshold) ,
}).round(3)

,our model,sklearn
roc_auc,0.704,0.702
pr_auc,0.312,0.307
accuracy,0.621,0.619
precision,0.266,0.265
recall,0.693,0.696
f1,0.384,0.383


### Both give almost the same results.

## Comparison

In [12]:
logistic = pd.read_parquet('../../data/ctr/logistic_test.parquet')
base = np.full(len(y_test_np) , y_train_np.mean())

def all_scores(y , p , pred , prob=True):
    s = {'log loss' : log_loss(y , p) if prob else np.nan}
    s.update(scores(y , p , pred))
    return s

pd.DataFrame({
    'baseline (train ctr)'  : all_scores(y_test_np , base , base >= 0.5) ,
    'logistic'              : all_scores(y_test_np , logistic['p'] , logistic['pred']) ,
    'logistic (weighted)'   : all_scores(y_test_np , logistic['pw'] , logistic['pred_w']) ,
    'svm (weighted)'        : all_scores(y_test_np , z_test , z_test >= threshold , prob=False) ,
}).round(3)

,baseline (train ctr),logistic,logistic (weighted),svm (weighted)
log loss,0.456,0.409,0.611,NaN
roc_auc,0.500,0.725,0.724,0.704
pr_auc,0.170,0.347,0.345,0.312
accuracy,0.830,0.655,0.651,0.621
precision,0.000,0.285,0.283,0.266
recall,0.000,0.679,0.683,0.693
f1,0.000,0.401,0.400,0.384


In [13]:
# roc curves put on a grid of 200 false positive rates , so the plot stays small
grid = np.linspace(0 , 1 , 200)
curves = []
for name , s in [('logistic' , logistic['p']) , ('logistic (weighted)' , logistic['pw']) , ('svm (weighted)' , z_test)]:
    fpr , tpr , _ = roc_curve(y_test_np , s)
    curves.append(pd.DataFrame({'fpr' : grid , 'tpr' : np.interp(grid , fpr , tpr) , 'model' : name}))
fig = px.line(pd.concat(curves) , x='fpr' , y='tpr' , color='model' , title='ROC curve (test)')
fig.add_shape(type='line' , x0=0 , y0=0 , x1=1 , y1=1 , line_dash='dash')
fig.show()

### Logistic regression is best on everything that matters here : lowest log loss (0.409) , highest ROC-AUC (0.725) and PR-AUC (0.347).
### The weighted logistic ranks the same but its probabilities are pushed up (log loss 0.611 , calibration plot in 03).
### The SVM ranks worse (0.704) and gives no probability , so no log loss.
### All three beat the baseline. Best model : plain logistic regression.

## 4. Inference and Interpretation
### Weights are in 03 : site and app categories , specific sites and apps , and banner position matter most.
### Ranking : the 10% rated highest click at 40% , the lowest 10% at 2.5% (calibration plot in 03).
### Predictive , not causal : ads are not placed at random , advertisers and publishers choose where they go. A high CTR in position 1 can just mean better ads get position 1.
### Only an A/B test that randomises placement shows whether moving an ad changes its clicks.
### Limitations : 1 in 32 rows , rare levels merged , anonymised features , 10 days with a random split , no feature combinations , no user history.

## 5. Recommendations
### 1. Bid by expected value , P(click) x price per click , using the plain logistic model. Its probabilities are calibrated.
### 2. Put the budget on the top of the ranking : the top 30% click at about 31% , the bottom 30% at about 5%.
### 3. Favour website inventory (20% vs 12% in apps) , connection type 0 and device type 0. Cut connection types 3 / 5 , device types 4 / 5 and the weakest site categories first. Test each move with an A/B test , since these are associations.
### 4. Don't bid with a class weighted model.
### 5. Next : all 40 million rows , models that learn feature combinations , user-history features , and validating on the last day.